# 🤖 BaristaBot — Agent conversationnel avec LangGraph et l'API Gemini

**Défi quotidien** : construire une application graphique à état avec LangGraph, alimentée par l'API Gemini.

## Ce que vous allez créer
`BaristaBot` est un système de commande conversationnel pour café. Il permet de :
- prendre des commandes en langage naturel,
- consulter un menu dynamique via un outil,
- confirmer et modifier les commandes,
- boucler la conversation jusqu'à validation,
- gérer les appels d'outils via `ToolNode`.

## Sommaire
1. Installation
2. Clé API
3. État (`OrderState`) et instruction système
4. Chatbot à tour unique
5. Visualisation du graphe
6. Exécution
7. Second tour de conversation
8. Nœud humain
9. Arête conditionnelle (sortie de boucle)
10. Menu dynamique (outil sans état)
11. Gestion des commandes (outils à état)
12. Graphe complet et exécution


---
## 1. Installation

Installez le SDK LangGraph et le support LangChain pour l'API Gemini.

In [ ]:
%pip install -qU "langgraph==1.0.5" "langchain-google-genai==4.1.2" "google-genai==1.56.0"

---
## 2. Configuration de la clé API

La variable d'environnement `GOOGLE_API_KEY` configure automatiquement l'API sous-jacente,
aussi bien pour le SDK Gemini officiel que pour LangChain/LangGraph.

Récupérez une clé sur [AI Studio](https://aistudio.google.com/app/apikey) si vous n'en avez pas.

> **Note Colab vs Kaggle** : le notebook d'origine utilise les *Kaggle Secrets*.
> La cellule ci-dessous fonctionne dans les deux environnements et retombe sur une
> saisie manuelle masquée si aucun gestionnaire de secrets n'est disponible.

In [ ]:
import os

def _load_api_key() -> str:
    # 1) Déjà présent dans l'environnement ?
    if os.environ.get("GOOGLE_API_KEY"):
        return os.environ["GOOGLE_API_KEY"]

    # 2) Google Colab : Secrets (icône 🔑 dans la barre latérale)
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass

    # 3) Kaggle : Add-ons > Secrets
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore
        return UserSecretsClient().get_secret("GOOGLE_API_KEY")
    except Exception:
        pass

    # 4) Repli : saisie manuelle masquée
    from getpass import getpass
    return getpass("GOOGLE_API_KEY : ")


os.environ["GOOGLE_API_KEY"] = _load_api_key()
print("Clé API configurée ✅")

---
## 3. État et instruction système

Une application LangGraph s'articule autour d'un **graphe** :

- Chaque **nœud** représente une action (appel LLM, appel d'API, logique métier). Un nœud
  reçoit l'état et renvoie le nouvel état : `état = nœud(état)`.
- Chaque **arête** représente une transition. Elle peut être fixe ou **conditionnelle**
  (branchements type `if/else`, boucles type `while`).

L'objet d'état est un dictionnaire Python dont on décrit le schéma via `TypedDict`.

`OrderState` contient :
- `messages` : l'historique de conversation. L'annotation `add_messages` indique à LangGraph
  d'**ajouter** les messages renvoyés plutôt que de les remplacer ;
- `order` : la commande en cours (ici une simple liste de chaînes) ;
- `finished` : un drapeau signalant que la commande est passée.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph.message import add_messages


class OrderState(TypedDict):
    """State representing the customer's order conversation."""

    # The chat conversation. This preserves the conversation history
    # between nodes. The `add_messages` annotation indicates to LangGraph
    # that state is updated by appending returned messages, not replacing
    # them.
    messages: Annotated[list, add_messages]

    # The customer's in-progress order.
    order: list[str]

    # Flag indicating that the order is placed and completed.
    finished: bool

L'instruction système définit le comportement attendu du chatbot : le ton, le périmètre de
discussion, et les règles d'appel des différents outils.

In [ ]:
# The system instruction defines how the chatbot is expected to behave and includes
# rules for when to call different functions, as well as rules for the conversation, such
# as tone and what is permitted for discussion.
BARISTABOT_SYSINT = (
    "system",  # 'system' indicates the message is a system instruction.
    "You are a BaristaBot, an interactive cafe ordering system. A human will talk to you about the "
    "available products you have and you will answer any questions about menu items (and only about "
    "menu items - no off-topic discussion, but you can chat about the products and their history). "
    "The customer will place an order for 1 or more items from the menu, which you will structure "
    "and send to the ordering system after confirming the order with the human. "
    "\n\n"
    "Add items to the customer's order with add_to_order, and reset the order with clear_order. "
    "To see the contents of the order so far, call get_order (this is shown to you, not the user) "
    "Always confirm_order with the user (double-check) before calling place_order. Calling confirm_order will "
    "display the order items to the user and returns their response to seeing the list. Their response may contain modifications. "
    "Always verify and respond with drink and modifier names from the MENU before adding them to the order. "
    "If you are unsure a drink or modifier matches those on the MENU, ask a question to clarify or redirect. "
    "You only have the modifiers listed on the menu. "
    "Once the customer has finished ordering items, Call confirm_order to ensure it is correct then make "
    "any necessary updates and then call place_order. Once place_order has returned, thank the user and "
    "say goodbye!",
)

# This is the message with which the system opens the conversation.
WELCOME_MSG = "Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?"

---
## 4. Un chatbot à tour unique

Pour illustrer le fonctionnement de LangGraph, on définit un nœud `chatbot` qui exécute un
seul tour de conversation.

> Le nœud renvoie `{"messages": [...]}`. Grâce à l'annotation `add_messages`, ce message est
> **ajouté** à l'historique au lieu de l'écraser.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

# Try using different models. The `pro` models perform the best, especially
# with tool-calling. The `flash` models are super fast, and are a good choice
# if you need to use the higher free-tier quota.
# Check out the features and quota differences here: https://ai.google.dev/pricing
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-latest")


def chatbot(state: OrderState) -> OrderState:
    """The chatbot itself. A simple wrapper around the model's own chat interface."""
    message_history = [BARISTABOT_SYSINT] + state["messages"]
    return {"messages": [llm.invoke(message_history)]}


# Set up the initial graph based on our state definition.
graph_builder = StateGraph(OrderState)

# Add the chatbot function to the app graph as a node called "chatbot".
graph_builder.add_node("chatbot", chatbot)

# Define the chatbot node as the app entrypoint.
graph_builder.add_edge(START, "chatbot")

chat_graph = graph_builder.compile()

---
## 5. Visualiser le graphe

In [ ]:
from IPython.display import Image

Image(chat_graph.get_graph().draw_mermaid_png())

---
## 6. Exécuter le graphe

Le graphe ne comporte qu'un nœud : il transite de `START` vers `chatbot`, exécute le nœud,
puis se termine. On l'invoque avec `invoke()` en lui passant un état initial.

In [ ]:
from pprint import pprint

user_msg = "Hello, what can you do?"
state = chat_graph.invoke({"messages": [user_msg]})

# The state object contains lots of information. Uncomment the pprint lines to see it all.
# pprint(state)

# Note that the final state now has 2 messages. Our HumanMessage, and an additional AIMessage.
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content}")

---
## 7. Un second tour de conversation

On reprend l'état issu du premier appel et on y ajoute un nouveau message utilisateur pour
susciter une nouvelle réponse.

In [ ]:
user_msg = "Oh great, what kinds of latte can you make?"

state["messages"].append(user_msg)
state = chat_graph.invoke(state)

# pprint(state)
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content}")

---
## 8. Ajouter un nœud humain

Plutôt que de boucler manuellement en Python, on laisse LangGraph boucler entre les nœuds.

Le nœud `human` affiche le dernier message du modèle puis attend une saisie utilisateur. Ici on
utilise `print`/`input`, mais dans un vrai café ce serait un écran, un micro ou un clavier tactile.

Le nœud `chatbot` est également mis à jour pour émettre le message de bienvenue au démarrage.

In [ ]:
from langchain_core.messages.ai import AIMessage


def human_node(state: OrderState) -> OrderState:
    """Display the last model message to the user, and receive the user's input."""
    last_msg = state["messages"][-1]
    print("Model:", last_msg.content)

    user_input = input("User: ")

    # If it looks like the user is trying to quit, flag the conversation
    # as over.
    if user_input in {"q", "quit", "exit", "goodbye"}:
        state["finished"] = True

    return state | {"messages": [("user", user_input)]}


def chatbot_with_welcome_msg(state: OrderState) -> OrderState:
    """The chatbot itself. A wrapper around the model's own chat interface."""

    if state["messages"]:
        # If there are messages, continue the conversation with the Gemini model.
        new_output = llm.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        # If there are no messages, start with the welcome message.
        new_output = AIMessage(content=WELCOME_MSG)

    return state | {"messages": [new_output]}


# Start building a new graph.
graph_builder = StateGraph(OrderState)

# Add the chatbot and human nodes to the app graph.
graph_builder.add_node("chatbot", chatbot_with_welcome_msg)
graph_builder.add_node("human", human_node)

# Start with the chatbot again.
graph_builder.add_edge(START, "chatbot")

# The chatbot will always go to the human next.
graph_builder.add_edge("chatbot", "human");

---
## 9. Sortir de la boucle : l'arête conditionnelle

Une arête simple `human -> chatbot` boucle à l'infini. Il faut une **condition de sortie**.

Dans LangGraph on utilise `add_conditional_edges`. La fonction de routage reçoit l'état et
renvoie le **nom du nœud** de destination (`END` pour terminer).

In [ ]:
from typing import Literal


def maybe_exit_human_node(state: OrderState) -> Literal["chatbot", "__end__"]:
    """Route to the chatbot, unless it looks like the user is exiting."""
    if state.get("finished", False):
        return END
    else:
        return "chatbot"


graph_builder.add_conditional_edges("human", maybe_exit_human_node)

chat_with_human_graph = graph_builder.compile()

Image(chat_with_human_graph.get_graph().draw_mermaid_png())

Exécutez la boucle conversationnelle. Tapez `q` pour quitter.

> ⚠️ Le nœud `human` utilise `input()` : la cellule reste en attente tant que vous n'avez pas
> répondu. Pensez à limiter la récursion pour éviter les boucles infinies.

In [ ]:
# The default recursion limit is 25; raise it for longer conversations.
config = {"recursion_limit": 100}

# Uncomment to run the interactive loop:
# state = chat_with_human_graph.invoke({"messages": []}, config)
# pprint(state)

---
## 10. Un menu dynamique

BaristaBot ignore aujourd'hui les produits réellement disponibles : il hallucine un menu par
défaut. On pourrait coder le menu en dur dans l'instruction système, mais pour simuler un menu
**dynamique** (variations de stock), on passe par un **outil**.

Deux familles d'outils cohabitent :

| Type | Exemple | Exécuté par | Modifie l'état ? |
|---|---|---|---|
| **Sans état** (auto) | `get_menu` | `ToolNode` (automatique) | Non |
| **Avec état** | `add_to_order`, `place_order`… | `order_node` (manuel) | Oui |

`get_menu` est sans état : on peut l'appeler automatiquement.
On annote une fonction Python avec `@tool` pour en faire un outil.

In [ ]:
from langchain_core.tools import tool


@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    # Note that this is just hard-coded text, but you could connect this to a live stock
    # database, or you could use Gemini's multi-modal capabilities and take live photos of
    # your cafe's chalk menu or the products on the counter and assemble them into an input.

    return """
    MENU:
    Coffee Drinks:
    Espresso
    Americano
    Cold Brew

    Coffee Drinks with Milk:
    Latte
    Cappuccino
    Cortado
    Macchiato
    Mocha
    Flat White

    Tea Drinks:
    English Breakfast Tea
    Green Tea
    Earl Grey

    Tea Drinks with Milk:
    Chai Latte
    Matcha Latte
    London Fog

    Other Drinks:
    Steamer
    Hot Chocolate

    Modifiers:
    Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free; Default option: whole
    Espresso shots: Single, Double, Triple, Quadruple; default: Double
    Caffeine: Decaf, Regular; default: Regular
    Hot-Iced: Hot, Iced; Default: Hot
    Sweeteners (option to add one or more): vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener
    Special requests: any reasonable modification that does not involve items not on the menu, for example: 'extra hot', 'one pump', 'half caff', 'extra foam', etc.

    "dirty" means add a shot of espresso to a drink that doesn't usually have it, like "Dirty Chai Latte".
    "Regular milk" is the same as 'whole milk'.
    "Sweetened" means add some regular sugar, not a sweetener.

    Soy milk has run out of stock today, so soy is not available.
  """

On ajoute l'outil au graphe. `get_menu` est enveloppé dans un `ToolNode` qui gère son appel et
la propagation de la réponse sous forme de `ToolMessage`. Les outils sont aussi **liés** au LLM
via `bind_tools` pour que le modèle sache ce qu'il peut appeler.

In [ ]:
from langgraph.prebuilt import ToolNode


# Define the tools and create a "tools" node.
tools = [get_menu]
tool_node = ToolNode(tools)

# Attach the tools to the model so that it knows what it can call.
llm_with_tools = llm.bind_tools(tools)


def maybe_route_to_tools(state: OrderState) -> Literal["tools", "human"]:
    """Route between human or tool nodes, depending if a tool call is made."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    # Only route based on the last message.
    msg = msgs[-1]

    # When the chatbot returns tool_calls, route to the "tools" node.
    if hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        return "tools"
    else:
        return "human"


def chatbot_with_tools(state: OrderState) -> OrderState:
    """The chatbot with tools. A simple wrapper around the model's own chat interface."""
    defaults = {"order": [], "finished": False}

    if state["messages"]:
        new_output = llm_with_tools.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)

    # Set up some defaults if not already set, then pass through the provided state,
    # overriding only the "messages" field.
    return defaults | state | {"messages": [new_output]}


graph_builder = StateGraph(OrderState)

# Add the nodes, including the new tool_node.
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)

# Chatbot may go to tools, or human.
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)
# Human may go back to chatbot, or exit.
graph_builder.add_conditional_edges("human", maybe_exit_human_node)

# Tools always route back to chat afterwards.
graph_builder.add_edge("tools", "chatbot")

graph_builder.add_edge(START, "chatbot")
graph_with_menu = graph_builder.compile()

Image(graph_with_menu.get_graph().draw_mermaid_png())

---
## 11. Gestion des commandes

Pour construire une commande au fil de la conversation, il faut mettre à jour l'état. Ces mises
à jour passent par des outils **explicites** : le modèle ne doit jamais manipuler directement
l'état interne de l'application.

LangGraph n'autorise pas un `@tool` à modifier l'état de conversation. On définit donc les outils
de commande comme des **fonctions vides** (le décorateur `@tool` sert uniquement à déclarer le
*schéma* transmis au LLM), et leur implémentation réelle vit dans `order_node`.

In [ ]:
from collections.abc import Iterable
from random import randint

from langchain_core.messages.tool import ToolMessage

# These functions have no body; LangGraph does not allow @tools to update
# the conversation state, so you will implement a separate node to handle
# state updates. Using @tools is still very convenient for defining the tool
# schema, so empty functions have been defined that will be bound to the LLM
# but their implementation is deferred to the order_node.


@tool
def add_to_order(drink: str, modifiers: Iterable[str]) -> str:
    """Adds the specified drink to the customer's order, including any modifiers.

    Returns:
      The updated order in progress.
    """


@tool
def confirm_order() -> str:
    """Asks the customer if the order is correct.

    Returns:
      The user's free-text response.
    """


@tool
def get_order() -> str:
    """Returns the users order so far. One item per line."""


@tool
def clear_order():
    """Removes all items from the user's order."""


@tool
def place_order() -> int:
    """Sends the order to the barista for fulfillment.

    Returns:
      The estimated number of minutes until the order is ready.
    """

`order_node` lit le **dernier message** (celui qui porte les `tool_calls`), exécute chaque appel
d'outil, et renvoie un `ToolMessage` par appel.

In [ ]:
def order_node(state: OrderState) -> OrderState:
    """The ordering node. This is where the order state is manipulated."""
    tool_msg = state.get("messages", [])[-1]
    order = state.get("order", [])
    outbound_msgs = []
    order_placed = False

    for tool_call in tool_msg.tool_calls:

        if tool_call["name"] == "add_to_order":

            # Each order item is just a string. This is where it assembled as "drink (modifiers, ...)".
            modifiers = tool_call["args"].get("modifiers", [])
            modifier_str = ", ".join(modifiers) if modifiers else "no modifiers"

            order.append(f'{tool_call["args"]["drink"]} ({modifier_str})')
            response = "\n".join(order)

        elif tool_call["name"] == "confirm_order":

            # We could entrust the LLM to do order confirmation, but it is a good practice to
            # show the user the exact data that comprises their order so that what they confirm
            # precisely matches the order that goes to the kitchen - avoiding hallucination
            # or reality skew.

            # In a real scenario, this is where you would connect your POS screen to show the
            # order to the user.

            print("Your order:")
            if not order:
                print("  (no items)")

            for drink in order:
                print(f"  {drink}")

            response = input("Is this correct? ")

        elif tool_call["name"] == "get_order":

            response = "\n".join(order) if order else "(no order)"

        elif tool_call["name"] == "clear_order":

            order.clear()
            response = None

        elif tool_call["name"] == "place_order":

            order_text = "\n".join(order)
            print("Sending order to kitchen!")
            print(order_text)

            # TODO(you!): Implement cafe.
            order_placed = True
            response = randint(1, 5)  # ETA in minutes

        else:
            raise NotImplementedError(f'Unknown tool call: {tool_call["name"]}')

        # Record the tool results as tool messages.
        outbound_msgs.append(
            ToolMessage(
                content=response,
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": outbound_msgs, "order": order, "finished": order_placed}

La fonction de routage devient plus riche : elle distingue les outils **automatiques**
(nœud `tools`) des outils de **commande** (nœud `ordering`), et termine le graphe une fois la
commande passée.

In [ ]:
def maybe_route_to_tools(state: OrderState) -> str:
    """Route between chat and tool nodes if a tool call is made."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    msg = msgs[-1]

    if state.get("finished", False):
        # When an order is placed, exit the app. The system instruction indicates
        # that the chatbot should say thanks and goodbye at this point, so we can exit
        # cleanly.
        return END

    elif hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        # Route to `tools` node for any automated tool calls first.
        if any(
            tool["name"] in tool_node.tools_by_name.keys() for tool in msg.tool_calls
        ):
            return "tools"
        else:
            return "ordering"

    else:
        return "human"

---
## 12. Le graphe complet

On déclare deux ensembles d'outils, correspondant aux deux nœuds qui les exécutent. Le LLM,
lui, doit connaître **tous** les outils : on les lie donc tous ensemble via `bind_tools`.

In [ ]:
# Auto-tools will be invoked automatically by the ToolNode
auto_tools = [get_menu]
tool_node = ToolNode(auto_tools)

# Order-tools will be handled by the order node.
order_tools = [add_to_order, confirm_order, get_order, clear_order, place_order]

# The LLM needs to know about all of the tools, so specify everything here.
llm_with_tools = llm.bind_tools(auto_tools + order_tools)


graph_builder = StateGraph(OrderState)

# Nodes
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("ordering", order_node)

# Chatbot -> {ordering, tools, human, END}
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)
# Human -> {chatbot, END}
graph_builder.add_conditional_edges("human", maybe_exit_human_node)

# Tools (both kinds) always route back to chat afterwards.
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("ordering", "chatbot")

graph_builder.add_edge(START, "chatbot")
graph_with_order_tools = graph_builder.compile()

Image(graph_with_order_tools.get_graph().draw_mermaid_png())

### Exécution

Lancez le système de commande complet.

**Choses à essayer :**
- Commander une boisson 🍵
- Modifier votre commande
- « Which teas are from England? »
- Demander du lait de soja (rupture de stock !)

Le graphe se termine naturellement après le passage de la commande.

In [ ]:
# The default recursion limit for traversing nodes is 25 - setting it higher
# means you can try a more complex order with multiple steps and round-trips.
config = {"recursion_limit": 100}

state = graph_with_order_tools.invoke({"messages": []}, config)

# Things to try:
# - Order a drink!
# - Make a change to your order.
# - "Which teas are from England?"
# - Note that the graph should naturally exit after placing an order.

pprint(state)

---
## 🎉 Bravo !

Vous avez construit un agent conversationnel à état avec LangGraph :

- un **schéma d'état** (`TypedDict` + `add_messages`),
- des **nœuds** (`chatbot`, `human`, `tools`, `ordering`),
- des **arêtes conditionnelles** pour le branchement et les boucles,
- deux familles d'**outils** : sans état (auto) et avec état (commande).

### Pour aller plus loin
- Remplacer `order: list[str]` par une structure Pydantic typée.
- Brancher `get_menu` sur une vraie base de stock.
- Ajouter un `checkpointer` (`MemorySaver`) pour persister les conversations.
- Essayer un modèle `pro` : l'appel d'outils y est plus fiable.